# 290.407 and 291.407 on 2020-09-23 and 2020-09-24

These two mice ran their first two days in the DLS rig's date-organized MED-PC files, and 290.407's CSV exports of those days sit in `DLS Excitatory/ChR2/290.407`, so the published files label the runs DLS Excitatory (ChR2 or Unknown). The demographics sheet and all of their sessions from 2020-09-25 on put them in DMS Excitatory: 290.407 with stimulation on random nose pokes, 291.407 as a control. This notebook reconverts the four runs with the fixed labels and compares them with the published copies.

In [1]:
from pathlib import Path
import shutil

import h5py
import numpy as np
import pandas as pd

from lerner_lab_to_nwb.seiler_2024.seiler_2024_convert_dataset import (
    opto_to_nwb,
    get_no_port_entry_duration_sessions,
    get_session_key_from_kwargs,
)
from lerner_lab_to_nwb.seiler_2024.seiler_2024_convert_session import session_to_nwb

data_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/_Raw_Data_for_future_reference")
published_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/000971")
output_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/fix_audit/dms_excitatory_relabel")
if output_dir_path.exists():
    shutil.rmtree(output_dir_path)
output_dir_path.mkdir(parents=True)

In [2]:
selected_kwargs = [
    session_to_nwb_kwargs
    for session_to_nwb_kwargs in opto_to_nwb(
        data_dir_path=data_dir_path, output_dir_path=output_dir_path, start_variable="Start Date", verbose=False
    )
    if session_to_nwb_kwargs["subject_id"] in ("290.407", "291.407")
    and session_to_nwb_kwargs["session_conditions"]["Start Date"] in ("09/23/20", "09/24/20")
]
no_port_entry_duration_sessions = get_no_port_entry_duration_sessions(
    port_entry_file_path=output_dir_path / "sessions_without_port_entry_durations.yaml",
    session_to_nwb_args_per_session=selected_kwargs,
)
for session_to_nwb_kwargs in selected_kwargs:
    if get_session_key_from_kwargs(session_to_nwb_kwargs) in no_port_entry_duration_sessions:
        session_to_nwb_kwargs["has_port_entry_durations"] = False
    session_to_nwb(**session_to_nwb_kwargs)

Reading port entry durations:   0%|          | 0/4 [00:00<?, ?it/s]

Reading port entry durations: 100%|██████████| 4/4 [00:00<00:00, 79.17it/s]

In [3]:
def summarize(file_path: Path) -> dict:
    with h5py.File(file_path, "r") as file:
        summary = dict(
            file=file_path.name.split("_ses-")[1],
            stimulus_notes=file["general/stimulus"][()].decode() if "general/stimulus" in file else "",
            stimulation_site=(
                file["general/optogenetics/OptogeneticStimulusSite/location"][()].decode().split("Stimulation location: ")[1]
                if "general/optogenetics/OptogeneticStimulusSite" in file
                else ""
            ),
        )
        behavior_datasets = {}
        file["processing/behavior"].visititems(
            lambda name, item: behavior_datasets.__setitem__(name, item[()]) if isinstance(item, h5py.Dataset) else None
        )
    return summary, behavior_datasets


rows = []
for new_file_path in sorted(output_dir_path.glob("*.nwb")):
    subject_id = new_file_path.name.split("_ses-")[0].removeprefix("sub-")
    session_stamp = new_file_path.stem[-19:]
    new_summary, new_datasets = summarize(new_file_path)
    rows.append(dict(subject_id=subject_id, version="fixed", **new_summary, events_equal_to_fixed=True))
    for published_file_path in sorted((published_dir_path / f"sub-{subject_id.replace('.', '-')}").glob(f"*{session_stamp}*.nwb")):
        published_summary, published_datasets = summarize(published_file_path)
        events_equal = set(published_datasets) == set(new_datasets) and all(
            np.array_equal(published_datasets[name], new_datasets[name]) for name in new_datasets
        )
        rows.append(dict(subject_id=subject_id, version="published", **published_summary, events_equal_to_fixed=events_equal))
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(rows)

,subject_id,version,file,stimulus_notes,stimulation_site,events_equal_to_fixed
0,290.407,fixed,Opto-DMS-Excitatory-ChR2Scrambled-2020-09-23T12-41-13.nwb,Excitatory stimulation on random nosepokes,"DMS (AP 0.8, ML 1.5, DV -2.8)",True
1,290.407,published,Opto-DLS-Excitatory-ChR2-2020-09-23T12-41-13_ogen.nwb,Excitatory stimulation on random nosepokes,"DLS (AP -0.1, ML 2.8, DV -3.5)",True
2,290.407,published,Opto-DLS-Excitatory-Unknown-2020-09-23T12-41-13_ogen.nwb,Excitatory stimulation on random nosepokes,"DLS (AP -0.1, ML 2.8, DV -3.5)",True
3,290.407,fixed,Opto-DMS-Excitatory-ChR2Scrambled-2020-09-24T12-22-41.nwb,Excitatory stimulation on random nosepokes,"DMS (AP 0.8, ML 1.5, DV -2.8)",True
4,290.407,published,Opto-DLS-Excitatory-ChR2-2020-09-24T12-22-41_ogen.nwb,Excitatory stimulation on random nosepokes,"DLS (AP -0.1, ML 2.8, DV -3.5)",True
5,290.407,published,Opto-DLS-Excitatory-Unknown-2020-09-24T12-22-41_ogen.nwb,Excitatory stimulation on random nosepokes,"DLS (AP -0.1, ML 2.8, DV -3.5)",True
6,291.407,fixed,Opto-DMS-Excitatory-EYFP-2020-09-23T12-41-13.nwb,Control,"DMS (AP 0.8, ML 1.5, DV -2.8)",True
7,291.407,published,Opto-DLS-Excitatory-Unknown-2020-09-23T12-41-13_ogen.nwb,Control,"DLS (AP -0.1, ML 2.8, DV -3.5)",True
8,291.407,fixed,Opto-DMS-Excitatory-EYFP-2020-09-24T12-22-41.nwb,Control,"DMS (AP 0.8, ML 1.5, DV -2.8)",True
9,291.407,published,Opto-DLS-Excitatory-Unknown-2020-09-24T12-22-41_ogen.nwb,Control,"DLS (AP -0.1, ML 2.8, DV -3.5)",True
